**Task 2:** VisRet (ColSmol page-image retriever), VisRAG-top1, VisRAG-top3.

**Task 3:** NoRetrieval, Oracle-text, Oracle-image.

In [ ]:
!pip install -q datasets huggingface_hub rank_bm25 sentence-transformers
!pip install -q transformers accelerate qwen-vl-utils colpali-engine
!pip install -q "torchao>=0.16.0"

## 1. Config

In [ ]:
import os, re, json, string, random, math, time, glob
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

SET_NAME = "MP-DocVQA"            # "MP-DocVQA" or "InfoVQA"
REPO = {"MP-DocVQA": "openbmb/VisRAG-Ret-Test-MP-DocVQA",
        "InfoVQA":   "openbmb/VisRAG-Ret-Test-InfoVQA"}[SET_NAME]
CACHE = f"/content/drive/MyDrive/visrag/{SET_NAME}"
os.makedirs(CACHE, exist_ok=True)
N_QUESTIONS, SEED = 200, 0

# ONE prompt for ALL systems
PROMPT = ("Answer the question using the provided document. "
          "Reply with only the short answer, no explanation.\n"
          "Question: {q}")

Mounted at /content/drive


## 2. Data (loaded from Drive)

In [ ]:
import csv
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

def load_beir_qrels(path):
    qrels = {}
    with open(path) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])
    return qrels

corpus  = load_dataset(REPO, name="corpus",  split="train")
queries = load_dataset(REPO, name="queries", split="train")
qf = [f for f in list_repo_files(REPO, repo_type="dataset")
      if f.startswith("qrels/") and f.endswith(".tsv")]
assert len(qf) == 1, qf
qrels_raw = load_beir_qrels(hf_hub_download(REPO, qf[0], repo_type="dataset"))

print(corpus[0].keys(), queries[0])
page_ids = [str(r["corpus-id"]) for r in corpus]
qrels = {qid: {pid for pid, s in d.items() if s > 0} for qid, d in qrels_raw.items()}
q_all = [{"qid": str(r["query-id"]), "question": r["query"],
          "answers": r["answer"] if isinstance(r["answer"], list) else [r["answer"]]}
         for r in queries]

sub_path = f"{CACHE}/subset_qids.json"
if os.path.exists(sub_path):
    keep = set(json.load(open(sub_path)))
else:
    random.seed(SEED)
    keep = {q["qid"] for q in random.sample(q_all, min(N_QUESTIONS, len(q_all)))}
    json.dump(sorted(keep), open(sub_path, "w"))
q_sub = [q for q in q_all if q["qid"] in keep]
print(len(page_ids), "pages |", len(q_all), "questions |", len(q_sub), "in subset")
print("qrels pages found in corpus:", len({p for s in qrels.values() for p in s} & set(page_ids)))

README.md:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

corpus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

corpus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/741 [00:00<?, ? examples/s]

queries/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 40.2kB            

queries/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/591 [00:00<?, ? examples/s]

docvqa_mp-eval-qrels.tsv:   0%|          | 0.00/71.2k [00:00<?, ?B/s]

dict_keys(['corpus-id', 'image']) {'query-id': 'txpp0227_p9.jpg-1', 'query': 'Who is ‘presiding’ TRRF GENERAL SESSION (PART 1)?', 'answer': ['lee a. waller', 'TRRF Vice President'], 'options': None, 'is_numerical': 0}
741 pages | 591 questions | 200 in subset
qrels pages found in corpus: 741


In [ ]:
import torch, gc, math, string, re
page_idx = {p: i for i, p in enumerate(page_ids)}
ocr_text = json.load(open(f"{CACHE}/ocr.json"))
text_rank = json.load(open(f"{CACHE}/best_text_rank.json"))
sub_ids = [q["qid"] for q in q_sub]
all_ids = [q["qid"] for q in q_all]

def retrieval_metrics(rank, qrels, qids):
    r1 = r5 = mrr = ndcg = 0.0
    for qid in qids:
        rel, top = qrels[qid], rank[qid]
        r1 += any(d in rel for d in top[:1])
        r5 += any(d in rel for d in top[:5])
        for k, d in enumerate(top[:10]):
            if d in rel:
                mrr += 1 / (k + 1); break
        dcg  = sum(1 / math.log2(k + 2) for k, d in enumerate(top[:5]) if d in rel)
        idcg = sum(1 / math.log2(k + 2) for k in range(min(len(rel), 5)))
        ndcg += dcg / idcg
    n = len(qids)
    return {"Recall@1": r1/n, "Recall@5": r5/n, "MRR@10": mrr/n, "nDCG@5": ndcg/n}

def norm(s):
    s = s.lower()
    s = "".join(c for c in s if c not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def correct(pred, golds):
    return norm(pred) in {norm(g) for g in golds}

def page_image(pid): return corpus[page_idx[pid]]["image"].convert("RGB")

## 3. VisRet: ColSmol-500M page-image retriever (no OCR)

In [ ]:
import torch
import time
import gc
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

# Ensure the crashed GPU memory is fully cleared first
gc.collect()
torch.cuda.empty_cache()

VNAME = "vidore/colSmol-500M"
torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
vmodel = ColIdefics3.from_pretrained(VNAME, torch_dtype=torch_dtype, device_map="cuda").eval()
vproc = ColIdefics3Processor.from_pretrained(VNAME)

def trim(e):                       # drop zero (padding) rows
    return e[e.abs().sum(-1) > 0].cpu()

@torch.no_grad()
def embed_pages(bs=4): # Reduced to 4 to prevent T4 GPU Out-Of-Memory
    path = f"{CACHE}/colsmol_pages.pt"
    if os.path.exists(path): return torch.load(path)
    out, t0 = [], time.time()
    for b in range(0, len(page_ids), bs):
        # 1. Measure and load images
        t_img = time.time()
        imgs = [page_image(p) for p in page_ids[b:b+bs]]
        t_img_done = time.time() - t_img

        # 2. Process and run inference (with AMP mixed precision for speed)
        batch = vproc.process_images(imgs).to(vmodel.device)
        with torch.cuda.amp.autocast(dtype=torch_dtype):
            e = vmodel(**batch)
        out.extend(trim(x) for x in e)

        # 3. Clean up GPU fragmentation & memory
        del batch, e
        gc.collect()
        torch.cuda.empty_cache()

        # 4. Print detailed metrics instantly with flush=True
        print(f"Batch {b//bs + 1}/{math.ceil(len(page_ids)/bs)} | "
              f"{len(out)}/{len(page_ids)} pages | "
              f"Img Load: {t_img_done:.2f}s | "
              f"Total: {(time.time()-t0)/60:.1f} min", end="\r", flush=True)
    torch.save(out, path)
    return out

@torch.no_grad()
def embed_queries(bs=16):
    out = []
    for b in range(0, len(q_all), bs):
        batch = vproc.process_queries([q["question"] for q in q_all[b:b+bs]]).to(vmodel.device)
        with torch.cuda.amp.autocast(dtype=torch_dtype):
            out.extend(trim(x) for x in vmodel(**batch))
    return out

page_embs = embed_pages()
q_embs = embed_queries()
print("\npage embedding example:", page_embs[0].shape, "| query:", q_embs[0].shape)

Loading weights:   0%|          | 0/490 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

/tmp/ipykernel_1771/2665279273.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch_dtype):


/tmp/ipykernel_1771/2665279273.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch_dtype):



page embedding example: torch.Size([1139, 128]) | query: torch.Size([29, 128])


In [ ]:
@torch.no_grad()
def maxsim_matrix(Q, P, page_bs=4, q_bs=100, dev="cuda"):
    """Scores [n_queries, n_pages]: sum over query tokens of max over page tokens."""
    S = torch.zeros(len(Q), len(P))
    Lq = max(x.shape[0] for x in Q)
    for qs in range(0, len(Q), q_bs):
        qc = Q[qs:qs+q_bs]
        Qp = torch.zeros(len(qc), Lq, qc[0].shape[1], dtype=torch.float16, device=dev)
        for i, x in enumerate(qc): Qp[i, :x.shape[0]] = x.to(dev)
        for ps in range(0, len(P), page_bs):
            pc = P[ps:ps+page_bs]; Lp = max(x.shape[0] for x in pc)
            Pp = torch.zeros(len(pc), Lp, pc[0].shape[1], dtype=torch.float16, device=dev)
            mask = torch.zeros(len(pc), Lp, dtype=torch.bool, device=dev)
            for i, x in enumerate(pc):
                Pp[i, :x.shape[0]] = x.to(dev); mask[i, :x.shape[0]] = True
            sim = torch.einsum("qnd,bpd->qbnp", Qp, Pp)
            sim = sim.masked_fill(~mask[None, :, None, :], -1e4)
            S[qs:qs+len(qc), ps:ps+len(pc)] = sim.max(-1).values.float().sum(-1).cpu()
    return S.numpy()

S = maxsim_matrix(q_embs, page_embs)
np.save(f"{CACHE}/visret_scores.npy", S)           # rows = q_all order, cols = page_ids order
vis_rank = {q["qid"]: [page_ids[i] for i in np.argsort(-S[j])[:10]] for j, q in enumerate(q_all)}
json.dump(vis_rank, open(f"{CACHE}/visret_rank.json", "w"))

res = {"VisRet": {"subset200": retrieval_metrics(vis_rank, qrels, sub_ids),
                  "all": retrieval_metrics(vis_rank, qrels, all_ids)}}
res["BM25/BGE (best text, Task 1)"] = {"subset200": retrieval_metrics(text_rank, qrels, sub_ids)}
json.dump(res, open(f"{CACHE}/retrieval_vis.json", "w"), indent=2)
for n, m in res.items(): print(n, {k: round(v, 3) for k, v in m["subset200"].items()})
assert res["VisRet"]["subset200"]["Recall@5"] > 0.3, "VisRet looks broken"

del vmodel, vproc, page_embs, q_embs
gc.collect(); torch.cuda.empty_cache()

VisRet {'Recall@1': 0.86, 'Recall@5': 0.975, 'MRR@10': 0.908, 'nDCG@5': 0.924}
BM25/BGE (best text, Task 1) {'Recall@1': 0.62, 'Recall@5': 0.78, 'MRR@10': 0.687, 'nDCG@5': 0.704}


## 4. Generator (shared with Task 1)

In [ ]:
def load_generator():
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    name = "Qwen/Qwen2.5-VL-3B-Instruct"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto")      # T4: fp16, not bf16
    proc = AutoProcessor.from_pretrained(
        name, min_pixels=256*28*28, max_pixels=1024*28*28)       # lower max_pixels if OOM
    return model, proc

def generate(model, proc, question, texts=None, images=None, max_new_tokens=32):
    content = []
    for t in (texts or []):
        content.append({"type": "text", "text": f"Document:\n{t}"})
    for im in (images or []):
        content.append({"type": "image", "image": im})
    content.append({"type": "text", "text": PROMPT.format(q=question)})
    prompt = proc.apply_chat_template([{"role": "user", "content": content}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[prompt], images=images or None, return_tensors="pt",
                  padding=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs.input_ids.shape[1]:],
                             skip_special_tokens=True)[0].strip()

def run_system(model, proc, tag, build_inputs):
    """Generic runner: build_inputs(q) -> dict(texts=..., images=...). Cached per question."""
    path = f"{CACHE}/gen_{tag}.json"
    preds = json.load(open(path)) if os.path.exists(path) else {}
    for q in q_sub:
        if q["qid"] in preds: continue
        preds[q["qid"]] = generate(model, proc, q["question"], **build_inputs(q))
        if len(preds) % 20 == 0: json.dump(preds, open(path, "w"))
    json.dump(preds, open(path, "w"))
    acc = float(np.mean([correct(preds[q["qid"]], q["answers"]) for q in q_sub]))
    print(f"{tag}: accuracy = {acc:.3f}")
    return preds, acc

In [ ]:
model, proc = load_generator()

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 5. Run the systems

In [ ]:
accs = {}
def run(tag, build):
    _, accs[tag] = run_system(model, proc, tag, build)
    json.dump(accs, open(f"{CACHE}/acc_task23.json", "w"), indent=2)

gold_page = lambda q: sorted(qrels[q["qid"]])[0]

# Task 2
run("VisRAG-top1", lambda q: {"images": [page_image(p) for p in vis_rank[q["qid"]][:1]]})
run("VisRAG-top3", lambda q: {"images": [page_image(p) for p in vis_rank[q["qid"]][:3]]})
# Task 3
run("NoRetrieval",  lambda q: {})
run("Oracle-text",  lambda q: {"texts": [ocr_text[gold_page(q)]]})
run("Oracle-image", lambda q: {"images": [page_image(gold_page(q))]})

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


VisRAG-top1: accuracy = 0.785
VisRAG-top3: accuracy = 0.785
NoRetrieval: accuracy = 0.095
Oracle-text: accuracy = 0.590
Oracle-image: accuracy = 0.835


## 6. Summary table (MP-DocVQA)

In [ ]:
import os
import json
import pandas as pd

# Path to accuracy results
t1_path = f"{CACHE}/acc_task1.json"
t23_path = f"{CACHE}/acc_task23.json"

t1 = json.load(open(t1_path)) if os.path.exists(t1_path) else {}
t23 = json.load(open(t23_path)) if os.path.exists(t23_path) else {}

allacc = {**t1, **t23}
order = ["NoRetrieval", "TextRAG-top1", "TextRAG-top3", "VisRAG-top1", "VisRAG-top3", "Oracle-text", "Oracle-image"]

print(f"Dataset: {SET_NAME}\n")
results_df = pd.DataFrame({
    "Accuracy": {k: f"{allacc[k]:.3f}" for k in order if k in allacc}
})
display(results_df)

Dataset: MP-DocVQA



,Accuracy
NoRetrieval,0.095
TextRAG-top1,0.445
TextRAG-top3,0.455
VisRAG-top1,0.785
VisRAG-top3,0.785
Oracle-text,0.590
Oracle-image,0.835
